# AeroDepth-Lite: MidAir training on Kaggle
This notebook runs repository code. It prepares train/validation work and never evaluates final test data.
Attach the verified MidAir dataset and the separate project-code snapshot. Enable a Kaggle GPU.
ImageNet initialization needs Internet enabled on first use, or the official weights cached in the PyTorch cache.
All generated files go to `/kaggle/working`; dataset images and HDF5 files remain in `/kaggle/input`.
Review `docs/kaggle_training.md`, the trajectory inventory, depth range, and budget before enabling the full run.


In [ ]:
from pathlib import Path
import sys, json, platform, shutil
import torch, torchvision
WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)
ENVIRONMENT = {
    'python': platform.python_version(), 'torch': torch.__version__,
    'torchvision': torchvision.__version__, 'cuda_runtime': torch.version.cuda,
    'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    'storage_free_gb': shutil.disk_usage(WORK).free / 1e9,
}
print(json.dumps(ENVIRONMENT, indent=2))


## Load the project snapshot
Kaggle commonly extracts uploaded ZIPs. Both extracted snapshots and intact code ZIPs are supported.
If more than one snapshot is attached, set `CODE_SOURCE` explicitly. Copy the sidecar checksum into
`EXPECTED_BUNDLE_SHA256` when an intact ZIP is available. Extracted files are checked individually
against the bundled provenance; that does not verify the original ZIP checksum.
The bootstrap uses `--no-deps` and preserves the installed CUDA-enabled PyTorch and torchvision.


In [ ]:
import zipfile, runpy
INPUT = Path('/kaggle/input')
CODE_SOURCE = None  # Optional Path('/kaggle/input/project-code') or intact ZIP
EXPECTED_BUNDLE_SHA256 = None  # SHA-256 from the supplied .zip.sha256 sidecar
if CODE_SOURCE is None:
    candidates = [p.parent for p in INPUT.rglob('SOURCE_PROVENANCE.json')]
    candidates += list(INPUT.rglob('aerodepth_kaggle_code.zip'))
    if len(candidates) != 1:
        raise ValueError(f'Set CODE_SOURCE to one project snapshot; found {candidates}')
    CODE_SOURCE = candidates[0]
CODE_SOURCE = Path(CODE_SOURCE)
if CODE_SOURCE.is_file():
    if EXPECTED_BUNDLE_SHA256:
        import hashlib
        assert hashlib.sha256(CODE_SOURCE.read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256
    with zipfile.ZipFile(CODE_SOURCE) as archive:
        bootstrap_bytes = archive.read('scripts/kaggle_setup.py')
else:
    bootstrap_bytes = (CODE_SOURCE / 'scripts/kaggle_setup.py').read_bytes()
bootstrap_file = WORK / 'kaggle_setup.py'
bootstrap_file.write_bytes(bootstrap_bytes)
setup = runpy.run_path(str(bootstrap_file))
PROJECT = setup['stage_project'](CODE_SOURCE, WORK / 'aerodepth_project', EXPECTED_BUNDLE_SHA256)
print(setup['install_project'](PROJECT))
print(json.loads((PROJECT / 'SOURCE_PROVENANCE.json').read_text())['source_claim'])


## Discover and validate the attached dataset
Validation checks all pair identities, image dimensions, HDF5 camera paths, and navigation synchronization.
It does not calculate test metrics or test-depth statistics. Supply the verified expected trajectory list
(relative environment/weather/trajectory IDs) or count to detect an incomplete attachment. Without it,
validation establishes internal consistency and records the discovered inventory for review.


In [ ]:
from aerodepth_lite.config import load_config
from aerodepth_lite.kaggle import discover_dataset_root, validate_dataset, load_or_create_manifest
CONFIG = load_config(PROJECT / 'configs/mobilenet.yaml')
DATASET_ROOT = None  # Optional explicit Path for multiple dataset attachments
EXPECTED_TRAJECTORIES = None  # Optional exact list from the verified dataset inventory
EXPECTED_TRAJECTORY_COUNT = None
DATASET_ROOT = Path(DATASET_ROOT) if DATASET_ROOT else discover_dataset_root(INPUT)
RECORDS, VALIDATION = validate_dataset(DATASET_ROOT, EXPECTED_TRAJECTORIES, EXPECTED_TRAJECTORY_COUNT)
PREP = WORK / 'preparation'
PREP.mkdir(exist_ok=True)
(PREP / 'dataset_validation.json').write_text(json.dumps(VALIDATION, indent=2))
(PREP / 'environment.json').write_text(json.dumps(ENVIRONMENT, indent=2))
print(DATASET_ROOT)
print(json.dumps({'trajectories': VALIDATION['trajectories'], 'frames': len(RECORDS),
                  'expected_inventory_verified': VALIDATION['expected_inventory_verified']}, indent=2))


## Deterministic grouped split
Weather variants of an environment/path stay together. The climate digit is removed: IDs 0003/1003/2003/3003 share path 0003.
The pilot IDs 2003 and 2021 map to Kite paths 0003 and 0021, excluded from final test in every weather. Add any other inspected path groups below.
An attached existing manifest is validated for coverage, fingerprint, group leakage, and pilot exclusions.
Keep this manifest fixed across sessions; never regenerate it in response to validation results.


In [ ]:
EXISTING_MANIFEST = None  # Optional Path to a saved/attached grouped manifest
ADDITIONAL_INSPECTED_PATHS = set()  # For example {'Kite/trajectory_0005'}
MANIFEST = PREP / 'grouped_seed42.json'
SPLITS = load_or_create_manifest(RECORDS, MANIFEST, CONFIG['seed'],
    Path(EXISTING_MANIFEST) if EXISTING_MANIFEST else None, ADDITIONAL_INSPECTED_PATHS)
print(json.dumps({'splits': SPLITS['splits'], 'frame_counts': SPLITS['frame_counts'],
                  'pilot_exclusions': SPLITS['pilot_paths_excluded_from_test']}, indent=2))


## Train/validation smoke and measured throughput
Two epochs with three batches per split verify real-data GPU integration. The last epoch provides a
rough throughput estimate after warm-up. This smoke is not a benchmark or a final research result.
Local automated tests use `pretrained=False`; this Kaggle smoke verifies actual ImageNet initialization.


In [ ]:
from copy import deepcopy
from aerodepth_lite.train import train_model
from aerodepth_lite.kaggle import estimate_training_seconds
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU before running the real-data smoke')
SMOKE_CONFIG = deepcopy(CONFIG)
SMOKE_CONFIG['train'].update(epochs=2, max_train_batches=3, max_val_batches=3)
SMOKE_CONFIG['train']['lr_schedule']['t_max'] = 2
SMOKE_RUN = WORK / 'mobilenet_smoke'
SMOKE = train_model(SMOKE_CONFIG, DATASET_ROOT, MANIFEST, SMOKE_RUN)
THROUGHPUT = estimate_training_seconds(SMOKE, SPLITS['frame_counts'], CONFIG['train']['epochs'])
(PREP / 'throughput_estimate.json').write_text(json.dumps(THROUGHPUT, indent=2))
print(json.dumps(THROUGHPUT, indent=2))
print('Estimate includes measured train and validation batch work. Allow extra time for checkpoints, setup, and session variation.')


## Review candidate settings and launch resumable training
`CONFIG` starts from the separate MobileNet candidate configuration. Review valid depth limits in metres,
learning-rate schedule, resolution, batch size, and the 50-epoch budget. Keep `t_max` fixed when resuming
an interrupted budget. Scheduler/replay thresholds remain provisional and are unused in this training.
Set the two review fields and `RUN_FULL_TRAINING` only after reviewing the preparation outputs.
For a new Kaggle session, attach saved outputs, set `EXISTING_MANIFEST` earlier and `RESUME_CHECKPOINT`
below to the attached `last.pt`. Keep its companion `best.pt` beside it. New outputs are written to working.


In [ ]:
RUN_FULL_TRAINING = False
RESUME_CHECKPOINT = None  # Optional Path('/kaggle/input/prior-run/last.pt')
FULL_RUN = WORK / 'mobilenet_full'
# Edit candidate choices here, before the run:
# CONFIG['data']['max_depth'] = ...
# CONFIG['train']['epochs'] = ...
# CONFIG['train']['lr_schedule']['t_max'] = ...
# CONFIG['review']['depth_range_approved'] = True
# CONFIG['review']['training_budget_approved'] = True
if RUN_FULL_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError('Full training requires a GPU')
    if not (CONFIG['review']['depth_range_approved'] and CONFIG['review']['training_budget_approved']):
        raise ValueError('Review the depth range and measured training budget first')
    resume = Path(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else (
        FULL_RUN / 'last.pt' if (FULL_RUN / 'last.pt').exists() else None)
    FULL_SUMMARY = train_model(CONFIG, DATASET_ROOT, MANIFEST, FULL_RUN, resume=resume)
    print(json.dumps(FULL_SUMMARY, indent=2))
else:
    print('Prepared. Full training awaits the reviewed settings and RUN_FULL_TRAINING=True.')


## Validation only: metrics and plots
This uses the best validation-AbsRel checkpoint and the repository's existing losses and metrics.
The cell uses the smoke run unless a full run was launched. No final-test evaluation cell is included.


In [ ]:
from aerodepth_lite.kaggle import validation_outputs
RUN = FULL_RUN if RUN_FULL_TRAINING else SMOKE_RUN
RUN_CONFIG = load_config(RUN / 'config.yaml')
VAL_RESULTS = validation_outputs(RUN_CONFIG, DATASET_ROOT, MANIFEST, RUN / 'best.pt', RUN / 'validation')
print(json.dumps(VAL_RESULTS, indent=2))
from IPython.display import display, Image as DisplayImage
display(DisplayImage(filename=str(RUN / 'validation/loss_curve.png')))
display(DisplayImage(filename=str(RUN / 'validation/prediction_00.png')))


## Retain outputs
Download the run directory and `preparation` before the Kaggle session expires. Retain `best.pt`,
`last.pt` (latest epoch), `history.csv`, `config.yaml`, `manifest.json`, `metadata.json`, `run.json`,
validation outputs, the environment/throughput reports, and `code_bundle.json`.
Use `last.pt` plus its companion `best.pt` to resume. Checkpoints are at epoch boundaries; interruption
within an epoch repeats that epoch. Reattach the same dataset and manifest. Only load your own trusted checkpoints.
Final test execution is a separate future action after model and replay choices are frozen.
Actual Kaggle GPU execution has not been verified by the local preparation work.
